# 05_risk_scoring_model

This notebook builds a quantitative risk score for each ASRS incident using:

- HFACS levels  
- extracted human-factor indicators  
- contributing factors  
- operational risk signals  
- narrative characteristics  

The goal is to produce a reproducible, interpretable risk index suitable for safety analytics and modeling.


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/asrs_hfacs_3yrs.csv")

print("Loaded:", df.shape)
df.head()

Loaded: (16535, 135)


C:\Users\jenny\AppData\Local\Temp\ipykernel_9892\2795895245.py:1: DtypeWarning: Columns (7,8,15,19,20,38,39,40,41,42,43,44,45,46,47,48,49,50,59,63,78,79,81,82,83,86,89,99,100,110,111,123) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/asrs_hfacs_3yrs.csv")


,ACN,Date,Local Time Of Day,Locale Reference,State Reference,Relative Position.Angle.Radial,Relative Position.Distance.Nautical Miles,Altitude.AGL.Single Value,Altitude.MSL.Single Value,Latitude / Longitude (UAS),...,source_file,TimeOfDayBucket,causal_chain,contributing_factors,human_factors,risk_signals,HFACS_unsafe_acts,HFACS_preconditions,HFACS_unsafe_supervision,HFACS_organizational_influences
0,1714400,2020-01,0001-0600,ORD.Airport,IL,NaN,NaN,NaN,15000.0,NaN,...,ASRS_DBOnline_012020-062020.csv,Night,"['Contains temporal marker: during', 'Contains...",['workload'],['decision_errors'],NaN,False,False,False,False
1,1714553,2020-01,1801-2400,ZHU.ARTCC,TX,NaN,NaN,NaN,14000.0,NaN,...,ASRS_DBOnline_012020-062020.csv,Evening,['Contains temporal marker: after'],['automation'],['skill_based_errors'],NaN,True,False,False,False
2,1714675,2020-01,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,9000.0,NaN,...,ASRS_DBOnline_012020-062020.csv,Evening,"['Contains temporal marker: before', 'Contains...",['weather'],NaN,NaN,False,True,False,False
3,1714708,2020-01,0601-1200,ZZZ.Airport,US,NaN,NaN,0.0,NaN,NaN,...,ASRS_DBOnline_012020-062020.csv,Morning,['Contains temporal marker: before'],NaN,NaN,NaN,False,False,False,False
4,1714843,2020-01,1801-2400,OSU.Airport,OH,NaN,0.0,0.0,NaN,NaN,...,ASRS_DBOnline_012020-062020.csv,Evening,NaN,NaN,NaN,NaN,False,False,False,False


In [3]:
WEIGHTS = {
    # HFACS levels
    "HFACS_unsafe_acts": 3,
    "HFACS_preconditions": 2,
    "HFACS_unsafe_supervision": 2,
    "HFACS_organizational_influences": 1,

    # extracted human factors
    "decision_errors": 3,
    "skill_based_errors": 2,
    "situational_awareness": 2,
    "communication": 2,
    "fatigue": 2,
    "workload": 2,

    # contributing factors
    "weather": 1,
    "automation": 1,

    # operational risk signals
    "altitude deviation": 3,
    "loss of separation": 4,
    "near miss": 4,
    "unstable approach": 3,
}

In [4]:
def to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        return [x.lower()]
    return []

In [5]:
def compute_risk_score(row):
    score = 0

    # HFACS levels
    for col in ["HFACS_unsafe_acts", "HFACS_preconditions",
                "HFACS_unsafe_supervision", "HFACS_organizational_influences"]:
        if row.get(col) is True:
            score += WEIGHTS.get(col, 0)

    # human factors
    for hf in to_list(row.get("human_factors")):
        score += WEIGHTS.get(hf.lower(), 0)

    # contributing factors
    for cf in to_list(row.get("contributing_factors")):
        score += WEIGHTS.get(cf.lower(), 0)

    # risk signals
    for rs in to_list(row.get("risk_signals")):
        score += WEIGHTS.get(rs.lower(), 0)

    # narrative length bonus (longer narratives often indicate complexity)
    narrative = row.get("Narrative", "")
    if isinstance(narrative, str):
        length = len(narrative.split())
        if length > 150:
            score += 1
        if length > 300:
            score += 2

    return score

In [6]:
df["risk_score"] = df.apply(compute_risk_score, axis=1)
df["risk_score"].describe()

count    16535.000000
mean         1.970487
std          1.983719
min          0.000000
25%          0.000000
50%          1.000000
75%          3.000000
max          8.000000
Name: risk_score, dtype: float64

In [9]:
def categorize(score):
    if score >= 7:
        return "High"
    elif score >= 3:
        return "Medium"
    else:
        return "Low"

df["risk_category"] = df["risk_score"].apply(categorize)
df["risk_category"].value_counts()

risk_category
Low       9796
Medium    6399
High       340
Name: count, dtype: int64

In [11]:
output_file = "../data/processed/asrs_risk_scored_3yrs.csv"
df.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Final shape:", df.shape)

Saved: ../data/processed/asrs_risk_scored_3yrs.csv
Final shape: (16535, 137)
